# 08 - Compare learned NVS and 3DGS methods

This notebook does not train anything. It loads finished artifacts and shows them together:

1. **LagerNVS** - learned Plucker-ray RGB prediction;
2. **VGGT-X MCMC-3DGS** - official optimized Gaussian baseline;
3. **Ours** - confidence-weighted adaptive 3DGS supervised by real views and trusted LagerNVS pixels.

Only methods with real output files are displayed. Missing experiments are reported clearly.

In [ ]:
%pip -q install pandas pillow matplotlib imageio imageio-ffmpeg

import shutil, subprocess, sys
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive", force_remount=False)
CODE_ROOT = Path("/content/Project_Thesis_code")
REPOSITORY = "https://github.com/katlit/Project_Thesis.git"
BRANCH = "codex/hq200-example-notebook"
if CODE_ROOT.exists() and not (CODE_ROOT / ".git").is_dir(): shutil.rmtree(CODE_ROOT)
command = (["git", "clone", "--depth", "1", "--branch", BRANCH, REPOSITORY, str(CODE_ROOT)]
           if not CODE_ROOT.exists() else ["git", "-C", str(CODE_ROOT), "pull", "--ff-only", "origin", BRANCH])
subprocess.run(command, check=True)
sys.path.insert(0, str(CODE_ROOT / "code"))
PROJECT_ROOT = Path("/content/drive/MyDrive/ITU/3D/Thesis")

import imageio.v2 as imageio
import numpy as np
import pandas as pd
from IPython.display import Video, display
DATASET="3DRealCar"; SCENE=None
manifest=pd.read_csv(PROJECT_ROOT/"data_processed/method_inputs/manifest.csv")
available=manifest.query("method == 'vggt' and split == 'train' and dataset == @DATASET")
SCENE=SCENE or sorted(available.scene.unique())[0]
paths={
 "LagerNVS": PROJECT_ROOT/"experiments/SyntheticViews/LagerNVS"/DATASET/SCENE/"images",
 "VGGT-X MCMC-3DGS": PROJECT_ROOT/"experiments/3DGS/VGGT_X_MCMC"/DATASET/SCENE/"orbit_frames",
 "Ours: confidence-weighted 3DGS": PROJECT_ROOT/"experiments/3DGS/LagerNVS_confidence_weighted"/DATASET/SCENE/"orbit_frames",
}
for name,path in paths.items(): print(("FOUND" if path.exists() else "MISSING"),name,path)

## Quantitative interpretation

Training-view PSNR, SSIM and silhouette IoU measure how well a method fits its inputs. They do not prove novel-view quality. The main ranking should use held-out real images with known cameras. Runtime, peak GPU memory and Gaussian count should be reported beside image metrics.

In [ ]:
metric_files={
 "VGGT-X MCMC-3DGS": PROJECT_ROOT/"experiments/3DGS/VGGT_X_MCMC"/DATASET/SCENE/"metrics.csv",
 "Ours": PROJECT_ROOT/"experiments/3DGS/LagerNVS_confidence_weighted"/DATASET/SCENE/"metrics.csv",
}
tables=[]
for method,path in metric_files.items():
    if path.is_file():
        table=pd.read_csv(path); table.insert(0,"method",method); tables.append(table)
if tables: display(pd.concat(tables,ignore_index=True))
else: print("No comparable metric files yet. Finish notebooks 06 and 07 first.")

## Side-by-side orbit

For a strict visual comparison, export the same number of frames from the same canonical front-to-front orbit. Do not compare unrelated camera paths as if they were pixel-aligned.

In [ ]:
from PIL import Image, ImageDraw
frame_sets={name: sorted(path.glob("view_*.png")) for name,path in paths.items() if path.is_dir()}
if len(frame_sets) != 3:
    print("Finish all three methods first. Available:", {name:len(files) for name,files in frame_sets.items()})
else:
    count=min(len(files) for files in frame_sets.values())
    output=PROJECT_ROOT/"experiments/Comparisons"/DATASET/SCENE; output.mkdir(parents=True,exist_ok=True)
    video=output/"three_method_closed_orbit.mp4"
    writer=imageio.get_writer(video,fps=15,codec="libx264",quality=8,macro_block_size=None)
    try:
        for index in range(count):
            panels=[(name,Image.open(files[index]).convert("RGB")) for name,files in frame_sets.items()]
            h=max(panel.height for _,panel in panels); w=sum(panel.width for _,panel in panels)+8*(len(panels)-1)
            canvas=Image.new("RGB",(w,h+32),"white"); draw=ImageDraw.Draw(canvas); x=0
            for name,panel in panels:
                canvas.paste(panel,(x,32)); draw.text((x+8,8),name,fill="black"); x+=panel.width+8
            writer.append_data(np.asarray(canvas))
    finally: writer.close()
    print("Saved:",video); display(Video(str(video),embed=True,width=1200,html_attributes="controls autoplay loop muted"))